In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F 
import torch.backends.cudnn as cudnn

import torchvision
import torchvision.transforms as transforms

import numpy as np 
import matplotlib.pyplot as plt 
import matplotlib.pylab as plt2
import os

from models import *

# from resnet50 import ResNet50
# from resnet34 import ResNet34

from torch.optim.lr_scheduler import ReduceLROnPlateau




In [2]:
#Check GPU, connect to it if it is available 
device = ''
if torch.cuda.is_available():
	device = 'cuda'
	print("CUDA is available. GPU will be used for training.")
else:
	device = 'cpu'


BEST_ACCURACY = 0

# Preparing Data
print("==> Prepairing data ...")
#Transformation on train data
transform_train = transforms.Compose([
	transforms.RandomCrop(32, padding=4),
	transforms.RandomHorizontalFlip(),
	transforms.ToTensor(),
	transforms.Normalize((0.4914, 0.4822, 0.4465),(0.2023, 0.1994, 0.2010)),
	])

#transformation on validation data
transform_validation = transforms.Compose([
	transforms.ToTensor(),
	transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
	])

#Download Train and Validation data and apply transformation
train_data = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
validation_data = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_validation)

#Put data into trainloader, specify batch_size
train_loader = torch.utils.data.DataLoader(train_data, batch_size=128, shuffle=True, num_workers=2)
validation_loader = torch.utils.data.DataLoader(validation_data, batch_size=128, shuffle=True, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

#Function to show CIFAR images
def show_data(image):
	plt.imshow(np.transpose(image[0], (1, 2, 0)), interpolation='bicubic')
	plt.show()

#show_data(train_data[0])


#Need to import model a model
model = ResNet50()
# model = ResNet34Cifar10(num_classes = 10)
# model = ResNet34()
#model = CNN_batch()
#Pass model to GPU
model = model.to(device)
model.train()
optimizer = optim.SGD(model.parameters(), lr = 0.01, momentum=0.9, weight_decay=5e-4)
criterion = nn.CrossEntropyLoss()
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=5, verbose=True)



length_train = len(train_data)
length_validation = len(validation_data)
#print(length_train)
#print(len(train_loader))
num_classes = 10

==> Prepairing data ...
Files already downloaded and verified
Files already downloaded and verified


In [3]:
#Training
def train(epochs):
	global BEST_ACCURACY
	dict = {'Train Loss':[], 'Train Acc':[], 'Validation Loss':[], 'Validation Acc':[]}
	for epoch in range(epochs):
		print("\nEpoch:", epoch+1, "/", epochs)
		cost = 0
		correct = 0
		total = 0
		woha = 0

		for i, (x,y) in enumerate(train_loader):
            
			woha += 1
			model.train()
			x, y = x.to(device), y.to(device)
			optimizer.zero_grad()
			yhat = model(x)
			yhat = yhat.reshape(-1, 10)
			loss = criterion(yhat, y)
			loss.backward()
			optimizer.step()
			cost += loss.item()

			_, yhat2 = torch.max(yhat.data, 1)
			correct += (yhat2 == y).sum().item()
			total += y.size(0)
			print("\nAcc:", correct/total, correct, "/", total)

		my_loss = cost/len(train_loader)
		my_accuracy = 100*correct/length_train

		dict['Train Loss'].append(my_loss)
		dict['Train Acc'].append(my_accuracy)

		print('Tain Loss:', my_loss)
		print('Train Accuracy:', my_accuracy,'%')


		cost = 0
		correct = 0

		with torch.no_grad():
			for x, y in validation_loader:
				x, y = x.to(device), y.to(device)
				model.eval()
				yhat = model(x)
				yhat = yhat.reshape(-1, 10)
				loss = criterion(yhat, y)
				cost += loss.item()
				
				_, yhat2 = torch.max(yhat.data, 1)
				correct += (yhat2 == y).sum().item()

		my_loss = cost/len(validation_loader)
		my_accuracy = 100*correct/length_validation

		dict['Validation Loss'].append(my_loss)
		dict['Validation Acc'].append(my_accuracy)

		print('Validation Loss:', my_loss)
		print('Validation Accuracy:', my_accuracy,'%')
        
		# Update the optimizer parameters
		optimizer.step()

		# Update the scheduler based on the validation accuracy
		scheduler.step(my_accuracy)

		# Zero the gradients
		optimizer.zero_grad()

		#Save the model if you get best accuracy on validation data
		if my_accuracy > BEST_ACCURACY:
			BEST_ACCURACY = my_accuracy
			print('Saving the model ...')
			model.eval()
			if not os.path.isdir('checkpoint'):
			    os.mkdir('checkpoint')
			torch.save(model.state_dict(), './checkpoint/resnet50_own.pth')

	print("TRAINING IS FINISHED !!!")
	return dict



In [ ]:
#Start training
results = train(70)


plt.figure(1)
plt.plot(results['Train Loss'], 'b', label = 'training loss')
plt.plot(results['Validation Loss'], 'r', label = 'validation loss')
plt.title("LOSS")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend(['training set', 'validation set'], loc='center right')
plt.savefig('Loss_ResNet50.png', dpi=300, bbox_inches='tight')

plt.figure(2)
plt.plot(results['Train Acc'], 'b', label = 'training accuracy')
plt.plot(results['Validation Acc'], 'r', label = 'validation accuracy')
plt.title("ACCURACY")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend(['training set', 'validation set'], loc='center right')
plt.savefig('Accuracy_ResNet50.png', dpi=300, bbox_inches='tight')
plt.show()
plt.close()

"""
axs[0].plot(results['Train Loss'], 'b', label = 'training loss')
axs[0].plot(results['Validation Loss'], 'r', label = 'validation loss')
axs[0].set_title("LOSS")
axs[0].set(xlabel="Epochs", ylabel="Loss")

axs[1].plot(results['Train Acc'], 'b', label = 'training accuracy')
axs[1].plot(results['Validation Acc'], 'r', label = 'validation accuracy')
axs[1].set_title("ACCURACY")
axs[1].set(xlabel="Epochs", ylabel="Accuracy")

fig.tight_layout()
plt.legend()
plt.show()
"""
# torch.save(model.backbone.state_dict(), './Results/resnet50/backbone_own.pth')
# torch.save(model.linear.state_dict(), './Results/resnet50/linear_own.pth')


Epoch: 1 / 70

Acc: 0.0703125 9 / 128

Acc: 0.1171875 30 / 256

Acc: 0.12239583333333333 47 / 384

Acc: 0.130859375 67 / 512

Acc: 0.125 80 / 640

Acc: 0.125 96 / 768

Acc: 0.1171875 105 / 896

Acc: 0.11328125 116 / 1024

Acc: 0.1189236111111111 137 / 1152

Acc: 0.11796875 151 / 1280

Acc: 0.11576704545454546 163 / 1408

Acc: 0.11588541666666667 178 / 1536

Acc: 0.11358173076923077 189 / 1664

Acc: 0.11104910714285714 199 / 1792

Acc: 0.1125 216 / 1920

Acc: 0.1103515625 226 / 2048

Acc: 0.10661764705882353 232 / 2176

Acc: 0.10416666666666667 240 / 2304

Acc: 0.10855263157894737 264 / 2432

Acc: 0.1140625 292 / 2560

Acc: 0.11532738095238096 310 / 2688

Acc: 0.11576704545454546 326 / 2816

Acc: 0.11888586956521739 350 / 2944

Acc: 0.12272135416666667 377 / 3072

Acc: 0.1234375 395 / 3200

Acc: 0.12319711538461539 410 / 3328

Acc: 0.1293402777777778 447 / 3456

Acc: 0.12918526785714285 463 / 3584

Acc: 0.12957974137931033 481 / 3712

Acc: 0.13125 504 / 3840

Acc: 0.13306451612903225 5


Acc: 0.27162388392857145 7788 / 28672

Acc: 0.2719097222222222 7831 / 28800

Acc: 0.27226216814159293 7876 / 28928

Acc: 0.2727147577092511 7924 / 29056

Acc: 0.2733347039473684 7977 / 29184

Acc: 0.27357396288209607 8019 / 29312

Acc: 0.27377717391304346 8060 / 29440

Acc: 0.27421536796536794 8108 / 29568

Acc: 0.2743803879310345 8148 / 29696

Acc: 0.2746445815450644 8191 / 29824

Acc: 0.27500667735042733 8237 / 29952

Acc: 0.2755651595744681 8289 / 30080

Acc: 0.2758540783898305 8333 / 30208

Acc: 0.27620648734177217 8379 / 30336

Acc: 0.2761292016806723 8412 / 30464

Acc: 0.2766409518828452 8463 / 30592

Acc: 0.27757161458333335 8527 / 30720

Acc: 0.2782676348547718 8584 / 30848

Acc: 0.27866735537190085 8632 / 30976

Acc: 0.27890303497942387 8675 / 31104

Acc: 0.2793609118852459 8725 / 31232

Acc: 0.27981505102040816 8775 / 31360

Acc: 0.28010670731707316 8820 / 31488

Acc: 0.28064903846153844 8873 / 31616

Acc: 0.28131300403225806 8930 / 31744

Acc: 0.2818461345381526 8983 / 3187


Acc: 0.47899305555555555 2759 / 5760

Acc: 0.48046875 2829 / 5888

Acc: 0.47938829787234044 2884 / 6016

Acc: 0.4793294270833333 2945 / 6144

Acc: 0.47847576530612246 3001 / 6272

Acc: 0.47828125 3061 / 6400

Acc: 0.4799325980392157 3133 / 6528

Acc: 0.4807692307692308 3200 / 6656

Acc: 0.4803950471698113 3259 / 6784

Acc: 0.4791666666666667 3312 / 6912

Acc: 0.4809659090909091 3386 / 7040

Acc: 0.4799107142857143 3440 / 7168

Acc: 0.4804002192982456 3505 / 7296

Acc: 0.48127693965517243 3573 / 7424

Acc: 0.4823887711864407 3643 / 7552

Acc: 0.48268229166666665 3707 / 7680

Acc: 0.48245389344262296 3767 / 7808

Acc: 0.48286290322580644 3832 / 7936

Acc: 0.48313492063492064 3896 / 8064

Acc: 0.482421875 3952 / 8192

Acc: 0.4829326923076923 4018 / 8320

Acc: 0.4844933712121212 4093 / 8448

Acc: 0.48484141791044777 4158 / 8576

Acc: 0.4846047794117647 4218 / 8704

Acc: 0.4852807971014493 4286 / 8832

Acc: 0.48660714285714285 4360 / 8960

Acc: 0.4873459507042254 4429 / 9088

Acc: 0.487304


Acc: 0.5221851145038168 17512 / 33536

Acc: 0.5221304657794676 17577 / 33664

Acc: 0.5223721590909091 17652 / 33792

Acc: 0.5222287735849057 17714 / 33920

Acc: 0.5222920582706767 17783 / 34048

Acc: 0.5226182116104869 17861 / 34176

Acc: 0.5226504197761194 17929 / 34304

Acc: 0.5230018587360595 18008 / 34432

Acc: 0.5230613425925926 18077 / 34560

Acc: 0.5233510147601476 18154 / 34688

Acc: 0.5237247242647058 18234 / 34816

Acc: 0.5239239926739927 18308 / 34944

Acc: 0.5238366788321168 18372 / 35072

Acc: 0.5239488636363636 18443 / 35200

Acc: 0.5238620923913043 18507 / 35328

Acc: 0.5241990072202166 18586 / 35456

Acc: 0.5243367805755396 18658 / 35584

Acc: 0.5245015681003584 18731 / 35712

Acc: 0.524609375 18802 / 35840

Acc: 0.524799822064057 18876 / 35968

Acc: 0.5251828457446809 18957 / 36096

Acc: 0.5253423144876325 19030 / 36224

Acc: 0.5255556778169014 19105 / 36352

Acc: 0.5258771929824562 19184 / 36480

Acc: 0.5260052447552448 19256 / 36608

Acc: 0.5262685104529616 19333 / 


Acc: 0.6208274147727273 6993 / 11264

Acc: 0.6209620786516854 7074 / 11392

Acc: 0.6214409722222223 7159 / 11520

Acc: 0.621823489010989 7243 / 11648

Acc: 0.6229619565217391 7336 / 11776

Acc: 0.6232358870967742 7419 / 11904

Acc: 0.6240026595744681 7508 / 12032

Acc: 0.6243421052631579 7592 / 12160

Acc: 0.6241861979166666 7670 / 12288

Acc: 0.6239529639175257 7747 / 12416

Acc: 0.6242028061224489 7830 / 12544

Acc: 0.6242108585858586 7910 / 12672

Acc: 0.6246875 7996 / 12800

Acc: 0.6243038366336634 8071 / 12928

Acc: 0.6235447303921569 8141 / 13056

Acc: 0.6237864077669902 8224 / 13184

Acc: 0.6240985576923077 8308 / 13312

Acc: 0.6235119047619048 8380 / 13440

Acc: 0.6241892688679245 8469 / 13568

Acc: 0.6241968457943925 8549 / 13696

Acc: 0.6246383101851852 8635 / 13824

Acc: 0.6247849770642202 8717 / 13952

Acc: 0.6248579545454546 8798 / 14080

Acc: 0.6255630630630631 8888 / 14208

Acc: 0.6252790178571429 8964 / 14336

Acc: 0.6254148230088495 9046 / 14464

Acc: 0.62623355263157

...


Acc: 0.9078799460431655 16153 / 17792

Acc: 0.9079799107142857 16271 / 17920

Acc: 0.9081338652482269 16390 / 18048

Acc: 0.9080105633802817 16504 / 18176

Acc: 0.9078343531468531 16617 / 18304

Acc: 0.9074978298611112 16727 / 18432

Acc: 0.9076508620689655 16846 / 18560

Acc: 0.907587756849315 16961 / 18688

Acc: 0.9078975340136054 17083 / 18816

Acc: 0.9080447635135135 17202 / 18944

Acc: 0.9080327181208053 17318 / 19072

Acc: 0.9079166666666667 17432 / 19200

Acc: 0.9078021523178808 17546 / 19328

Acc: 0.907946134868421 17665 / 19456

Acc: 0.9078329248366013 17779 / 19584

Acc: 0.9079748376623377 17898 / 19712

Acc: 0.9079637096774194 18014 / 19840

Acc: 0.9080028044871795 18131 / 19968

Acc: 0.9079916401273885 18247 / 20096

Acc: 0.907931170886076 18362 / 20224

Acc: 0.9082645440251572 18485 / 20352

Acc: 0.908154296875 18599 / 20480

Acc: 0.9080939440993789 18714 / 20608

Acc: 0.9077449845679012 18823 / 20736

Acc: 0.9080713190184049 18946 / 20864

Acc: 0.9077267530487805 19055 /


Acc: 0.9072931463068182 40879 / 45056

Acc: 0.9071795325779037 40990 / 45184

Acc: 0.9073313912429378 41113 / 45312

Acc: 0.9074383802816901 41234 / 45440

Acc: 0.9074569873595506 41351 / 45568

Acc: 0.9073441876750701 41462 / 45696

Acc: 0.9073193086592178 41577 / 45824

Acc: 0.9072728064066853 41691 / 45952

Acc: 0.9073133680555555 41809 / 46080

Acc: 0.9072238573407202 41921 / 46208

Acc: 0.9073290745856354 42042 / 46336

Acc: 0.9073906680440771 42161 / 46464

Acc: 0.9073231456043956 42274 / 46592

Acc: 0.907298801369863 42389 / 46720

Acc: 0.9072105532786885 42501 / 46848

Acc: 0.9072292234332425 42618 / 46976

Acc: 0.9070779551630435 42727 / 47104

Acc: 0.9070757113821138 42843 / 47232

Acc: 0.9070945945945946 42960 / 47360

Acc: 0.9071765498652291 43080 / 47488

Acc: 0.907174059139785 43196 / 47616

Acc: 0.9070459115281502 43306 / 47744

Acc: 0.9070228943850267 43421 / 47872

Acc: 0.9070416666666666 43538 / 48000

Acc: 0.9071226728723404 43658 / 48128

Acc: 0.9071825265251989 43


Acc: 0.9138879189944135 20939 / 22912

Acc: 0.9140190972222222 21059 / 23040

Acc: 0.9140625 21177 / 23168

Acc: 0.9143629807692307 21301 / 23296

Acc: 0.9142332650273224 21415 / 23424

Acc: 0.9143597146739131 21535 / 23552

Acc: 0.9143581081081081 21652 / 23680

Acc: 0.9141885080645161 21765 / 23808

Acc: 0.9143967245989305 21887 / 23936

Acc: 0.9143533909574468 22003 / 24064

Acc: 0.914227843915344 22117 / 24192

Acc: 0.9142269736842106 22234 / 24320

Acc: 0.9141443062827225 22349 / 24448

Acc: 0.9140218098958334 22463 / 24576

Acc: 0.9141029792746114 22582 / 24704

Acc: 0.9139416881443299 22695 / 24832

Acc: 0.9137820512820513 22808 / 24960

Acc: 0.9137834821428571 22925 / 25088

Acc: 0.9139038705583756 23045 / 25216

Acc: 0.9139046717171717 23162 / 25344

Acc: 0.9138662060301508 23278 / 25472

Acc: 0.9138671875 23395 / 25600

Acc: 0.9138681592039801 23512 / 25728

Acc: 0.9137144183168316 23625 / 25856

Acc: 0.9139085591133005 23747 / 25984

Acc: 0.9139859068627451 23866 / 26112

A


Acc: 0.91282 45641 / 50000
Tain Loss: 0.24966301239283797
Train Accuracy: 91.282 %
Validation Loss: 0.46992152250265773
Validation Accuracy: 86.46 %

Epoch: 17 / 70

Acc: 0.921875 118 / 128

Acc: 0.9375 240 / 256

Acc: 0.9270833333333334 356 / 384

Acc: 0.9140625 468 / 512

Acc: 0.9109375 583 / 640

Acc: 0.9088541666666666 698 / 768

Acc: 0.9040178571428571 810 / 896

Acc: 0.9072265625 929 / 1024

Acc: 0.9071180555555556 1045 / 1152

Acc: 0.903125 1156 / 1280

Acc: 0.9048295454545454 1274 / 1408

Acc: 0.908203125 1395 / 1536

Acc: 0.9104567307692307 1515 / 1664

Acc: 0.91015625 1631 / 1792

Acc: 0.9109375 1749 / 1920

Acc: 0.91259765625 1869 / 2048

Acc: 0.9140625 1989 / 2176

Acc: 0.9131944444444444 2104 / 2304

Acc: 0.9140625 2223 / 2432

Acc: 0.9125 2336 / 2560

Acc: 0.9144345238095238 2458 / 2688

Acc: 0.9158380681818182 2579 / 2816

Acc: 0.9167798913043478 2699 / 2944

Acc: 0.9173177083333334 2818 / 3072

Acc: 0.9165625 2933 / 3200

Acc: 0.9164663461538461 3050 / 3328

Acc: 0.915


Acc: 0.9182719748858448 25741 / 28032

Acc: 0.9183238636363636 25860 / 28160


In [ ]:
# Save the layers except the linear layer
layer_dict = {}
for name, layer in model.named_children():
    if name != 'linear':
        layer_dict[name] = layer.state_dict()

# Save the linear layer separately
linear_dict = model.linear.state_dict()

torch.save(layer_dict, './Results/resnet34/backbone_50_new.pth')
torch.save(linear_dict, './Results/resnet34/linear_50_new.pth')